Estudiante: Scarlett Cedeño

**Instalacion e importacion de bibliotecas**

In [4]:
# nltk para obtener la lista de stop words en espanol
# pandas para mostrar la matriz BoW como un DataFrame
!pip install nltk pandas --quiet


import re # Modulo de expresiones regulares, para eliminar puntuacion del texto
import nltk # Biblioteca usada unicamente para las stop words
import pandas as pd # para crear y mostrar DataFrames

# Descarga el paquete 'stopwords' de NLTK que contiene listas de palabras vacias
nltk.download('stopwords')
# funcion que permite acceder a las stop words ya descargadas
from nltk.corpus import stopwords

# Carga las stop words en espanol y las guarda como un set
stop_words = set(stopwords.words('spanish')) # Se usa un set porque la busqueda es mas rapida

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Documentos de entrada**

In [5]:
# Lista de corpus de entrada
# Cada elemento es un string que representa un documento independiente
documentos = [
    "El perro corre en el parque todos los dias.",
    "El gato duerme sobre la cama durante la tarde.",
    "El perro y el gato juegan juntos en el jardin."
]

**Preprocesamiento de texto**

In [6]:
def preprocesar(texto):
    texto = texto.lower() # Convierte todo a minusculas

    # Elimina la puntuacion
    #[^a-zñáéíóúü\s] estos caracteres no deseados se eliminan
    texto = re.sub(r'[^a-zñáéíóúü\s]', '', texto)

    # Divide el texto en una lista de tokens
    tokens = texto.split()

    # elimina los tokens que sean stop words en espanol
    tokens = [t for t in tokens if t not in stop_words]

    # Devuelve la lista final de tokens ya limpia y filtrada
    return tokens

In [7]:
# Creamos una lista de tokens por cada documento
documentos_tokenizados = [preprocesar(doc) for doc in documentos]

# Recorre la lista de documentos tokenizados junto con su indice
for i, tokens in enumerate(documentos_tokenizados):
    # Imprime el numero de documento y su lista de tokens resultante
    print(f"Documento {i+1}:", tokens)

Documento 1: ['perro', 'corre', 'parque', 'dias']
Documento 2: ['gato', 'duerme', 'cama', 'tarde']
Documento 3: ['perro', 'gato', 'juegan', 'juntos', 'jardin']


**Construccion del vocabulario**

In [8]:
def construir_vocabulario(documentos_tokenizados):
    vocabulario = set()
    # Recorrer cada documento y agregar sus palabras unicas al vocabulario
    for tokens in documentos_tokenizados:
        vocabulario.update(tokens)
    # Ordenar alfabeticamente
    return sorted(vocabulario)

vocabulario = construir_vocabulario(documentos_tokenizados)
print("Vocabulario:", vocabulario)
print("Tamano del vocabulario:", len(vocabulario))

Vocabulario: ['cama', 'corre', 'dias', 'duerme', 'gato', 'jardin', 'juegan', 'juntos', 'parque', 'perro', 'tarde']
Tamano del vocabulario: 11


**Construccion de la matriz Bag of Words**

In [11]:
def construir_bow(documentos_tokenizados, vocabulario, binarizar=False):
    # Lista que almacenara una fila por cada documento, al final, representara la matriz BoW completa
    matriz = []

    # Recorre cada documento del corpus
    for tokens in documentos_tokenizados:
        # Lista que almacena el conteo de cada palabra del vocabulario
        fila = [] #sera una fila de la matriz

        # Recorre el vocabulario completo en orden
        for palabra in vocabulario:
            # Cuenta cuantas veces aparece esta palabra del vocabulario
            conteo = tokens.count(palabra)

            # Si el parametro binarizar esta activado (True), se ignora la frecuencia exacta y solo se registra
            if binarizar:
                conteo = 1 if conteo > 0 else 0 # si la palabra aparece (1) o no aparece (0) en el documento

            # Agrega el conteo (o el valor binario) de esta palabra a la fila
            fila.append(conteo)

        # Una vez recorrido todo el vocabulario, la fila esta completa y se agrega como una nueva fila a la matriz general
        matriz.append(fila)

    # Devuelve la matriz completa: una lista de listas,
    return matriz # cada fila es un documento y cada columna es una palabra del vocabulario

**BoW con frecuencias**

In [10]:
# matriz BoW usando frecuencias (binarizar=False)
# Cada valor representa cuantas veces aparece esa palabra en ese documento
matriz_frecuencias = construir_bow(documentos_tokenizados, vocabulario, binarizar=False)

# Convierte la matriz en un DataFrame de Pandas para visualizarla como tabla
df_frecuencias = pd.DataFrame(
    matriz_frecuencias,
    columns=vocabulario, #asigna el nombre de cada palabra a su columna correspondiente
    index=[f"Documento {i+1}" for i in range(len(documentos))] #asigna una etiqueta a cada fila
)

df_frecuencias #DataFrame resultante con las frecuencias de cada palabra por documento

,cama,corre,dias,duerme,gato,jardin,juegan,juntos,parque,perro,tarde
Documento 1,0,1,1,0,0,0,0,0,1,1,0
Documento 2,1,0,0,1,1,0,0,0,0,0,1
Documento 3,0,0,0,0,1,1,1,1,0,1,0


**BoW binarizada (presencia/ausencia)**

In [12]:
# cada valor solo indica presencia (1) o ausencia (0) de la palabra, sin importar cuantas veces aparezca
matriz_binaria = construir_bow(documentos_tokenizados, vocabulario, binarizar=True)

# Convierte la matriz binaria en un DataFrame de Pandas
df_binaria = pd.DataFrame(
    matriz_binaria,
    columns=vocabulario,
    index=[f"Documento {i+1}" for i in range(len(documentos))]
)

# Muestra el DataFrame resultante con la representacion binaria del BoW
df_binaria

,cama,corre,dias,duerme,gato,jardin,juegan,juntos,parque,perro,tarde
Documento 1,0,1,1,0,0,0,0,0,1,1,0
Documento 2,1,0,0,1,1,0,0,0,0,0,1
Documento 3,0,0,0,0,1,1,1,1,0,1,0
